# SOL/USDT preparation for the one-day setups (Phase 13)

Prepares SOL/USDT as the primary asset for the approved one-day setups D1 and D2 (notebook 15), **before any outcome is computed**.

**Volatility and counts only.** No return after a signal, no mean, median or share of positive returns, and no relationship between any condition and price direction is computed.

**Steps**
1. Download SOL/USDT 15-minute history once, and report rows, date range and the largest gaps.
2. Typical 1-hour move of SOL by New York hour, for the development period (to 2023-12-31) and 2024 onward.
3. Development-period `rv_open` terciles for SOL, the share of Saturday entries, and the number of development trades D1 and D2 would make with SOL's own 2/3 threshold (one trade per signal day).

**Definitions** (unchanged from notebooks 12, 14 and 15)
- **1-hour moves:** non-overlapping 1-hour periods starting on the hour (UTC), log(open at period end / open at period start) from 15-minute candles, dropped if an endpoint candle is missing. Typical move = sample standard deviation (%). The New York hour and the period (development: start before 2024-01-01) are those of the period start.
- **US-open measures:** `rv_open` and `ret_open` use the 8 15-minute candles opening 09:00–10:45 New York time on signal day t, with close-to-previous-close returns. They are NaN unless all 8 returns exist.
- **Development signal days:** days t whose one-day trade would exit (day t + 2) on or before 2023-12-31. SOL's daily data starts 2020-08-11.
- **D1 and D2:** exactly as approved in notebook 15, including the Saturday-entry exclusion and one trade per signal day (`find_one_day_trades`). The `rv_open` threshold is SOL's own development 2/3 quantile, as approved for the replication.

**Note on the choice of SOL.** Notebook 15 named BTC as the primary asset and ETH/SOL as unchanged replication. Making SOL primary changes that plan before any outcome; the rules themselves are unchanged. Notebook 08 had already examined SOL's daily data (continuation setup and base rates), so the choice of asset is not independent of earlier daily results.

## Step 0: Imports and settings

In [1]:
import time
from pathlib import Path

import ccxt
import numpy as np
import pandas as pd

from cryptosense.data import load_processed, refresh_symbol, symbol_to_slug, timeframe_to_ms
from cryptosense.features import add_features
from cryptosense.regimes import add_regimes
from cryptosense.intraday import NY_TZ, lag_by_time
from cryptosense.backtest import find_one_day_trades

DATA_DIR = Path.cwd().parent / "data"
SYMBOL = "SOL/USDT"
SPLIT = pd.Timestamp("2024-01-01", tz="UTC")
DEV_LAST_EXIT = pd.Timestamp("2023-12-31", tz="UTC")
WINDOW_HOURS_NY = [9, 10]
CANDLES_IN_WINDOW = 8
EXCLUDED_ENTRY_WEEKDAY = "Saturday"
CANDLE_MS = timeframe_to_ms("15m")
HOUR_MS = timeframe_to_ms("1h")

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

## Step 1: Download SOL/USDT 15-minute history (once)

`refresh_symbol` fetches the full history, validates it, drops the incomplete candle and saves raw and processed files. It runs only if the processed 15-minute file does not exist yet, so re-running this notebook does not download again.

In [2]:
processed_15m = DATA_DIR / "processed" / f"{symbol_to_slug(SYMBOL)}_15m.csv"
if processed_15m.exists():
    print(f"Using existing file {processed_15m.name} (no download).")
else:
    exchange = ccxt.binance()
    start = time.perf_counter()
    result = refresh_symbol(exchange, SYMBOL, DATA_DIR, exchange.milliseconds(), timeframe="15m")
    elapsed = time.perf_counter() - start
    print(result)
    print(f"Download took {elapsed:.1f} s ({elapsed / 60:.1f} min)")

{'symbol': 'SOL/USDT', 'rows': 217197, 'last_date': datetime.date(2026, 10, 22), 'problems': {'gaps': 10, 'missing_days': 94, 'zero_volume_days': 10}}
Download took 78.6 s (1.3 min)


In [3]:
m15 = load_processed(SYMBOL, DATA_DIR, timeframe="15m")
print(f"candles: {len(m15):,}")
print(f"range:   {m15.index[0]} -> {m15.index[-1]}")

steps = m15["timestamp"].diff()
gap_steps = steps[steps > CANDLE_MS]
missing = (gap_steps // CANDLE_MS - 1).astype(int)
print(f"gaps: {len(gap_steps)} | missing candles in total: {missing.sum():,}")

gaps = pd.DataFrame({"gap_ended": gap_steps.index, "missing_candles": missing.to_numpy()})
gaps.nlargest(10, "missing_candles").reset_index(drop=True)

candles: 217,197
range:   2020-08-11 06:00:00+00:00 -> 2026-10-22 16:30:00+00:00
gaps: 10 | missing candles in total: 94


,gap_ended,missing_candles
0,2021-04-25 08:45:00+00:00,18
1,2021-08-13 06:30:00+00:00,18
2,2020-12-21 18:00:00+00:00,16
3,2021-04-20 04:30:00+00:00,10
4,2021-09-29 09:00:00+00:00,8
5,2021-03-06 03:30:00+00:00,6
6,2021-02-11 05:00:00+00:00,5
7,2023-03-24 14:00:00+00:00,5
8,2020-11-30 07:00:00+00:00,4
9,2020-12-25 03:00:00+00:00,4


## Step 2: Typical 1-hour move by New York hour, development and 2024 onward

In [4]:
open_15m = m15.set_index("timestamp")["open"]
first_ts, last_ts = int(open_15m.index[0]), int(open_15m.index[-1])
starts = np.arange(-(-first_ts // HOUR_MS) * HOUR_MS, last_ts - HOUR_MS + 1, HOUR_MS)
log_move = np.log(open_15m.reindex(starts + HOUR_MS).to_numpy() / open_15m.reindex(starts).to_numpy())

hours = pd.DataFrame({"log_move": log_move}, index=pd.to_datetime(starts, unit="ms", utc=True))
print(f"1-hour periods on the grid: {len(hours):,} | dropped (missing candle): {int(hours['log_move'].isna().sum()):,}")
hours = hours.dropna()
hours["hour_ny"] = hours.index.tz_convert(NY_TZ).hour
hours["period"] = np.where(hours.index < SPLIT, "development (to 2023)", "2024 onward")

def move_by_hour(frame):
    g = frame.groupby("hour_ny")["log_move"]
    table = pd.DataFrame({"n": g.size(), "typical_move_%": 100 * g.std()})
    table.loc["all hours"] = [len(frame), 100 * frame["log_move"].std()]
    return table

by_hour_ny = pd.concat({p: move_by_hour(f) for p, f in hours.groupby("period", sort=False)}, axis=1)
by_hour_ny.index.name = "hour_ny"
print(by_hour_ny.to_string(formatters={
    col: (lambda v: f"{v:,.0f}") if col[1] == "n" else (lambda v: f"{v:.3f}")
    for col in by_hour_ny.columns
}))

1-hour periods on the grid: 54,322 | dropped (missing candle): 34
          development (to 2023)                2024 onward               
                              n typical_move_%           n typical_move_%
hour_ny                                                                  
0                         1,233          1.309       1,026          0.769
1                         1,238          1.396       1,028          0.775
2                         1,232          1.360       1,023          0.679
3                         1,236          1.529       1,026          0.752
4                         1,236          1.464       1,026          0.806
5                         1,238          1.427       1,026          0.709
6                         1,238          1.423       1,026          0.735
7                         1,238          1.555       1,026          0.717
8                         1,236          1.808       1,026          0.870
9                         1,236          1.699

## Step 3: US-open terciles, Saturday share and D1/D2 development trade counts (counts only)

In [5]:
daily = add_regimes(add_features(load_processed(SYMBOL, DATA_DIR)))

log_close = np.log(m15["close"])
ret_15m = log_close - lag_by_time(log_close, "15min")
in_window = np.isin(m15.index.tz_convert(NY_TZ).hour, WINDOW_HOURS_NY)
w = pd.DataFrame({"r": ret_15m[in_window], "day": m15.index[in_window].normalize()})
per_day = w.groupby("day")["r"].agg(rv_open=lambda r: np.sqrt((r ** 2).sum()), ret_open="sum", returns="count")
per_day.loc[per_day["returns"] != CANDLES_IN_WINDOW, ["rv_open", "ret_open"]] = np.nan
per_day.index = per_day.index.as_unit(daily.index.unit)
daily = daily.join(per_day[["rv_open", "ret_open"]])

daily["exit_day"] = pd.Series(daily.index, index=daily.index).shift(-2)
daily["entry_weekday"] = pd.Series(daily.index + pd.Timedelta(days=1), index=daily.index).dt.day_name()
dev_mask = daily["exit_day"] <= DEV_LAST_EXIT
dev_rv = daily.loc[dev_mask, "rv_open"].dropna()
RV_OPEN_HIGH_SOL = dev_rv.quantile(2 / 3)

print(f"daily: {len(daily):,} days, {daily.index[0].date()} -> {daily.index[-1].date()}")
print(f"development signal days: {daily.index[dev_mask][0].date()} -> {daily.index[dev_mask][-1].date()} "
      f"({int(dev_mask.sum()):,} days; {len(dev_rv):,} with a complete US-open window)")
print(f"rv_open quantiles (SOL development): 1/3 = {dev_rv.quantile(1 / 3):.6f}, 2/3 = {RV_OPEN_HIGH_SOL:.6f}")
not_saturday = daily["entry_weekday"] != EXCLUDED_ENTRY_WEEKDAY
print(f"Saturday entries among development signal days: {int((dev_mask & ~not_saturday).sum())} "
      f"of {int(dev_mask.sum())} ({(dev_mask & ~not_saturday).sum() / dev_mask.sum():.4f})")

daily: 2,262 days, 2020-08-11 -> 2026-10-20
development signal days: 2020-08-11 -> 2023-12-29 (1,236 days; 1,234 with a complete US-open window)
rv_open quantiles (SOL development): 1/3 = 0.012285, 2/3 = 0.021055
Saturday entries among development signal days: 177 of 1236 (0.1432)


In [6]:
SETUPS_D = {
    "D1": {"inputs": ["trend", "rv_open", "ret_open"],
           "rule": lambda d: (d["trend"] == "up") & (d["rv_open"] > RV_OPEN_HIGH_SOL) & (d["ret_open"] > 0)},
    "D2": {"inputs": ["rv_open", "ret_open", "log_return"],
           "rule": lambda d: (d["rv_open"] > RV_OPEN_HIGH_SOL) & (d["ret_open"] < 0) & (d["log_return"] > 0)},
}

rows = []
for name, setup in SETUPS_D.items():
    usable = daily[setup["inputs"]].notna().all(axis=1)
    signal = usable & setup["rule"](daily).fillna(False).astype(bool) & not_saturday
    trade_signal_days = daily.index[[entry - 1 for entry, _ in find_one_day_trades(signal)]]
    rows.append({
        "setup": name,
        "development days": int(dev_mask.sum()),
        "excluded (Saturday entry)": int((dev_mask & ~not_saturday).sum()),
        "skipped (missing data)": int((dev_mask & not_saturday & ~usable).sum()),
        "signal days": int((dev_mask & signal).sum()),
        "trades (one per signal day)": int((daily.loc[trade_signal_days, "exit_day"] <= DEV_LAST_EXIT).sum()),
    })
sol_frequency = pd.DataFrame(rows).set_index("setup")
print(sol_frequency.to_string())

       development days  excluded (Saturday entry)  skipped (missing data)  signal days  trades (one per signal day)
setup                                                                                                               
D1                 1236                        177                      86           69                           69
D2                 1236                        177                       2           64                           64


## Results

### 15-minute data

Download: 78.6 s. 217,197 processed candles, 2020-08-11 06:00 → 2026-10-22 16:30 UTC. Validation: 10 gaps (94 missing candles), 10 zero-volume candles.

| Gap ended | Missing candles |
|---|---|
| 2021-04-25 08:45 | 18 |
| 2021-08-13 06:30 | 18 |
| 2020-12-21 18:00 | 16 |
| 2021-04-20 04:30 | 10 |
| 2021-09-29 09:00 | 8 |
| 2021-03-06 03:30 | 6 |
| 2021-02-11 05:00 | 5 |
| 2023-03-24 14:00 | 5 |
| 2020-11-30 07:00 | 4 |
| 2020-12-25 03:00 | 4 |

### Typical 1-hour move by New York hour (%)

54,322 1-hour periods on the grid, 34 dropped for a missing endpoint candle. n per hour: development 1,232–1,238; 2024 onward 1,023–1,028.

| Hour NY | Development (to 2023) | 2024 onward |
|---|---|---|
| 00 | 1.309 | 0.769 |
| 01 | 1.396 | 0.775 |
| 02 | 1.360 | 0.679 |
| 03 | 1.529 | 0.752 |
| 04 | 1.464 | 0.806 |
| 05 | 1.427 | 0.709 |
| 06 | 1.423 | 0.735 |
| 07 | 1.555 | 0.717 |
| 08 | 1.808 | 0.870 |
| 09 | 1.699 | 1.133 |
| 10 | 1.970 | 1.294 |
| 11 | 1.615 | 1.092 |
| 12 | 1.677 | 0.892 |
| 13 | 1.458 | 0.932 |
| 14 | 1.424 | 1.007 |
| 15 | 1.419 | 0.849 |
| 16 | 1.364 | 0.902 |
| 17 | 1.442 | 0.824 |
| 18 | 1.326 | 0.837 |
| 19 | 1.618 | 0.802 |
| 20 | 1.749 | 0.927 |
| 21 | 1.544 | 0.936 |
| 22 | 1.417 | 0.822 |
| 23 | 1.201 | 0.791 |
| All hours | 1.518 (n = 29,672) | 0.881 (n = 24,616) |

Highest hour: 10:00 New York in both periods (1.970 / 1.294).

### US-open terciles, Saturday share and development trade counts (SOL)

- Daily SOL/USDT: 2,262 days, 2020-08-11 → 2026-10-20.
- Development signal days (exit on or before 2023-12-31): 2020-08-11 → 2023-12-29, 1,236 days; 1,234 with a complete US-open window.
- `rv_open` development quantiles: 1/3 = 0.012285, 2/3 = 0.021055 (threshold for D1 and D2 on SOL).
- Saturday entries: 177 of 1,236 development signal days (0.1432).

| Setup | Development days | Excluded (Saturday entry) | Skipped (missing data) | Signal days | Trades (one per signal day) |
|---|---|---|---|---|---|
| D1 | 1,236 | 177 | 86 | 69 | 69 |
| D2 | 1,236 | 177 | 2 | 64 | 64 |